# 02 — Download actual Unsplash photos
The Lite archive contains metadata. This notebook follows `photo_image_url` from your local `photos.tsv000`. Defaults: 2,000 photos, a longest side of 1,024 px, JPEG quality 90 and a 2 GiB image-folder cap. It preserves the whole frame. No images are fetched until `DOWNLOAD = True`.

In [ ]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'configs/baseline.json').is_file() and (p / 'src/adaptive_styles').is_dir()), None)
if root is None:
    raise RuntimeError('Start Jupyter from research/adaptive-styles.')
sys.path.insert(0, str(root / 'src'))
from adaptive_styles.unsplash import select_photos, download_photos
print('Workspace:', root)

In [ ]:
COUNT = 2000
SIZE = 1024
SEED = 42
MAX_GIB = 2
IPHONE_ONLY = False
DOWNLOAD = False

source = root / 'data/unsplash-research-dataset-lite-latest/photos.tsv000'
rows = select_photos(source, count=COUNT, seed=SEED, iphone_only=IPHONE_ONLY)
collection = f'unsplash-lite-{SIZE}-q90'
destination = root / 'data/raw/references' / collection
manifest = root / 'data/metadata' / f'{collection}.jsonl'
print(f'Selected {len(rows)} photos. Image folder cap: {MAX_GIB} GiB.')
print('Destination:', destination)

Selection is a reproducible random subset, not a scene-balanced or quality-filtered dataset. Review it before training. `IPHONE_ONLY = True` filters camera metadata; it does not prove the photographs are unedited.

Set `DOWNLOAD = True` above and run the following cell. Interrupting is safe; rerun the same selection to resume completed files. The notebook keeps attribution, source URLs and hashes in a local JSONL manifest. A 403/429 response stops the run. The storage cap includes existing files in the destination and excludes filesystem overhead and the small metadata journal. Run one downloader at a time.

In [ ]:
if DOWNLOAD:
    result = download_photos(rows, destination, manifest, size=SIZE, max_bytes=int(MAX_GIB * 1024**3))
    print(result)
else:
    print('Plan only. Set DOWNLOAD = True to start.')